### Navie Bayes Implementation without Using Sklearn

### Imporiting Libraries

In [75]:
import csv
import random
import math
import pandas as pd
from collections import defaultdict

#### import dataset

In [76]:
dataset = pd.read_csv("https://raw.githubusercontent.com/sahdevsaini/Data-Set/main/pima-indians-diabetes.data.csv")

In [77]:
dataset.head()

,6,148,72,35,0,33.6,0.627,50,1
0,1,85,66,29,0,26.6,0.351,31,0
1,8,183,64,0,0,23.3,0.672,32,1
2,1,89,66,23,94,28.1,0.167,21,0
3,0,137,40,35,168,43.1,2.288,33,1
4,5,116,74,0,0,25.6,0.201,30,0


#### Function to convert dataset into Array list

In [78]:
split_ratio = 0.90
dataset  = dataset.values
dataset      

array([[1.00e+00, 8.50e+01, 6.60e+01, ..., 3.51e-01, 3.10e+01, 0.00e+00],
       [8.00e+00, 1.83e+02, 6.40e+01, ..., 6.72e-01, 3.20e+01, 1.00e+00],
       [1.00e+00, 8.90e+01, 6.60e+01, ..., 1.67e-01, 2.10e+01, 0.00e+00],
       ...,
       [5.00e+00, 1.21e+02, 7.20e+01, ..., 2.45e-01, 3.00e+01, 0.00e+00],
       [1.00e+00, 1.26e+02, 6.00e+01, ..., 3.49e-01, 4.70e+01, 1.00e+00],
       [1.00e+00, 9.30e+01, 7.00e+01, ..., 3.15e-01, 2.30e+01, 0.00e+00]])

#### Split dataset into Train & Test

In [79]:
import random

def split_dataset(dataset, split_ratio):
    # Calculate the size of the training set based on the split ratio
    train_size = int(len(dataset) * split_ratio)
    # Initialize an empty list to store the training set
    train_set = []
    # Make a copy of the dataset to ensure no data is lost
    copy = list(dataset)
    # Continue until the training set reaches the desired size
    while len(train_set) < train_size:
        # Generate a random index within the range of the copy list
        index = random.randrange(len(copy))
        # Remove the element at the randomly selected index from the copy list
        # and add it to the training set
        train_set.append(copy.pop(index))
    # Return a list containing the training set and the remaining data as the test set
    return [train_set, copy]


In [80]:
train_set, test_set = split_dataset(dataset, split_ratio)
print('Split {0} rows into train={1} and test={2} rows'.format(len(dataset), len(train_set), len(test_set)))

Split 767 rows into train=690 and test=77 rows


#### Separate data by class

In [82]:
def separate_by_class(dataset):
    # Initialize an empty dictionary to store separated data by class
    separated = {}
    # Iterate over each data point in the dataset
    for i in range(len(dataset)):
        # Extract the vector (row) from the dataset
        vector = dataset[i]
        # Check if the class label (last element of the vector) is already a key in the dictionary
        if vector[-1] not in separated:
            # If not, create a new list as the value for this class label
            separated[vector[-1]] = []
        # Append the vector to the list corresponding to its class label
        separated[vector[-1]].append(vector)
    # Return the dictionary containing separated data
    return separated


#### Calculate mean

In [84]:

def mean(numbers):
    return sum(numbers) / float(len(numbers))

#### Calculate standard deviation

In [86]:
def stdev(numbers):
    # Calculate the average of the numbers
    avg = mean(numbers)
    # Calculate the variance using the formula sum((x - mean)^2) / (n - 1)
    variance = sum([pow(x - avg, 2) for x in numbers]) / float(len(numbers) - 1)
    # Return the square root of the variance to get the standard deviation
    return math.sqrt(variance)


In [87]:
# Summarize dataset by class
def summarize_by_class(dataset):
    separated = separate_by_class(dataset)
    summaries = {}
    for class_value, instances in separated.items():
        summaries[class_value] = [(mean(attribute), stdev(attribute)) for attribute in zip(*instances)]
    return summaries

In [88]:
summaries = summarize_by_class(train_set)
summaries

{0.0: [(3.3619909502262444, 3.040199841251858),
  (110.6131221719457, 25.91486339687061),
  (68.00452488687783, 17.48948443055916),
  (20.006787330316744, 14.765819729555673),
  (71.75113122171946, 101.5459707836353),
  (30.420588235294126, 7.671867896595273),
  (0.43115384615384594, 0.3014972220720222),
  (31.273755656108598, 11.581112503358392),
  (0.0, 0.0)],
 1.0: [(4.931451612903226, 3.7771069822741254),
  (140.76209677419354, 32.07878482847455),
  (71.03629032258064, 20.343077027424147),
  (22.298387096774192, 17.138388776592947),
  (98.66935483870968, 131.29391707276798),
  (35.182661290322564, 7.374086217608953),
  (0.5468951612903226, 0.3731257468634985),
  (36.78629032258065, 10.85787338859037),
  (1.0, 0.0)]}

#### Calculate the probability density function using the Gaussian distribution formula

In [89]:
def calculate_probability(x, mean, stdev):
    # Check if the standard deviation is zero
    if stdev == 0:
        # Handle the case of zero standard deviation
        # If x is equal to the mean, return 1, otherwise return 0
        return 1 if x == mean else 0
    else:
        # Calculate the exponent term of the Gaussian distribution formula
        exponent = math.exp(-(math.pow(x - mean, 2) / (2 * math.pow(stdev, 2))))
        # Calculate the probability density function using the Gaussian distribution formula
        return (1 / (math.sqrt(2 * math.pi) * stdev)) * exponent


 #### Return the probabilities for each class

In [90]:
def calculate_class_probabilities(summaries, input_vector):
    # Initialize a dictionary to store the probabilities for each class
    probabilities = {}
    # Iterate over each class value and its associated summaries
    for class_value, class_summaries in summaries.items():
        # Initialize the probability for the current class to 1
        probabilities[class_value] = 1
        # Iterate over each attribute summary for the current class
        for i in range(len(class_summaries)):
            # Extract the mean and standard deviation for the current attribute
            mean, stdev = class_summaries[i]
            # Extract the value of the current attribute from the input vector
            x = input_vector[i]
            # Calculate the probability of the current attribute given its mean and standard deviation
            # and multiply it with the current class probability
            probabilities[class_value] *= calculate_probability(x, mean, stdev)
    # Return the probabilities for each class
    return probabilities


#### Predict the class for a given instance

In [92]:
def predict(summaries, input_vector):
    # Calculate the probabilities for each class given the input vector
    probabilities = calculate_class_probabilities(summaries, input_vector)
    # Initialize variables to track the best label and its probability
    best_label, best_prob = None, -1
    # Iterate over each class and its associated probability
    for class_value, probability in probabilities.items():
        # Check if the current class has a higher probability than the best one found so far
        if best_label is None or probability > best_prob:
            # Update the best label and its probability
            best_label = class_value
            best_prob = probability
    # Return the best label predicted for the input vector
    return best_label


#### Make predictions for a set of instances

In [94]:
def get_predictions(summaries, test_set):
    # Initialize an empty list to store the predictions
    predictions = []
    # Iterate over each instance in the test set
    for i in range(len(test_set)):
        # Predict the class label for the current instance using the summaries
        result = predict(summaries, test_set[i])
        # Append the predicted label to the list of predictions
        predictions.append(result)
    # Return the list of predictions for all instances in the test set
    return predictions


In [95]:
predictions = get_predictions(summaries, test_set)
predictions

[0.0,
 1.0,
 1.0,
 0.0,
 0.0,
 1.0,
 0.0,
 0.0,
 0.0,
 0.0,
 1.0,
 0.0,
 0.0,
 0.0,
 0.0,
 0.0,
 0.0,
 0.0,
 0.0,
 1.0,
 1.0,
 0.0,
 1.0,
 1.0,
 0.0,
 0.0,
 0.0,
 1.0,
 0.0,
 0.0,
 0.0,
 0.0,
 1.0,
 0.0,
 0.0,
 0.0,
 0.0,
 0.0,
 0.0,
 0.0,
 1.0,
 0.0,
 0.0,
 1.0,
 0.0,
 0.0,
 0.0,
 1.0,
 0.0,
 0.0,
 0.0,
 0.0,
 0.0,
 0.0,
 0.0,
 0.0,
 0.0,
 0.0,
 0.0,
 0.0,
 1.0,
 1.0,
 0.0,
 0.0,
 0.0,
 1.0,
 1.0,
 0.0,
 0.0,
 0.0,
 0.0,
 0.0,
 0.0,
 0.0,
 1.0,
 1.0,
 0.0]

In [96]:
test_set

[array([ 10.   , 139.   ,  80.   ,   0.   ,   0.   ,  27.1  ,   1.441,
         57.   ,   0.   ]),
 array([1.00e+00, 1.89e+02, 6.00e+01, 2.30e+01, 8.46e+02, 3.01e+01,
        3.98e-01, 5.90e+01, 1.00e+00]),
 array([  8.   , 176.   ,  90.   ,  34.   , 300.   ,  33.7  ,   0.467,
         58.   ,   1.   ]),
 array([ 2.   , 84.   ,  0.   ,  0.   ,  0.   ,  0.   ,  0.304, 21.   ,
         0.   ]),
 array([ 7.   , 62.   , 78.   ,  0.   ,  0.   , 32.6  ,  0.391, 41.   ,
         0.   ]),
 array([  5.   , 137.   , 108.   ,   0.   ,   0.   ,  48.8  ,   0.227,
         37.   ,   1.   ]),
 array([ 1.   , 80.   , 55.   ,  0.   ,  0.   , 19.1  ,  0.258, 21.   ,
         0.   ]),
 array([  0.   , 125.   ,  96.   ,   0.   ,   0.   ,  22.5  ,   0.262,
         21.   ,   0.   ]),
 array([ 2.  , 85.  , 65.  ,  0.  ,  0.  , 39.6 ,  0.93, 27.  ,  0.  ]),
 array([  1.   , 126.   ,  56.   ,  29.   , 152.   ,  28.7  ,   0.801,
         21.   ,   0.   ]),
 array([  5.  , 124.  ,  74.  ,   0.  ,   0.  ,  34.  

In [97]:
# Calculate the accuracy of predictions

In [98]:
def get_accuracy(test_set, predictions):
    # Initialize a variable to count the number of correct predictions
    correct = 0
    # Iterate over each instance in the test set
    for x in range(len(test_set)):
        # Check if the predicted label matches the actual label for the current instance
        if test_set[x][-1] == predictions[x]:
            # If the prediction is correct, increment the correct count
            correct += 1
    # Calculate the accuracy as the percentage of correct predictions
    accuracy = (correct / float(len(test_set))) * 100.0
    # Return the calculated accuracy
    return accuracy


In [99]:
accuracy = get_accuracy(test_set, predictions)
print('Accuracy: {0}%'.format(accuracy))

Accuracy: 100.0%
